# WSMTE — 05_ablation.ipynb
**KAGGLE GPU (T4 x2)** | Day 5 Tasks 5.1–5.12

Config H: PSO weight search (Stage 2) + 30-seed fine-tuning (Stage 3) + SHAP analysis.

**Kaggle input datasets required:**
- `wsmte-processed-data` — `/kaggle/input/wsmte-processed-data/`
- `wsmte-models` — `/kaggle/input/wsmte-models/config_g_best.keras`
  (upload `config_g_best.keras` from notebook 04 output)

In [ ]:
# ── Cell 1: Clone repo + sys.path + CONFIG ─────────────────────────────────
!git clone https://github.com/IAMNEERAJ05/WSMTE.git
import sys
sys.path.insert(0, '/kaggle/working/WSMTE')

from config.config import CONFIG
print(f"PSO particles  : {CONFIG['pso_n_particles']}")
print(f"PSO iterations : {CONFIG['pso_iterations']}")
print(f"PSO options    : c1={CONFIG['pso_c1']}, c2={CONFIG['pso_c2']}, w={CONFIG['pso_w']}")
print(f"Finetune epochs: {CONFIG['pso_finetune_epochs']}")
print(f"Config H runs  : {CONFIG['ablation_configs']['H']['n_runs']}")

## GPU Check

In [ ]:
# ── Cell 2: Verify GPU ────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np
import pandas as pd
import json

gpus = tf.config.list_physical_devices('GPU')
print(f'TF version : {tf.__version__}')
print(f'GPUs found : {len(gpus)}')
if not gpus:
    print('WARNING: No GPU detected.')

## Load All Data Arrays

In [ ]:
# ── Cell 3: Load processed data ───────────────────────────────────────────
DATA_DIR = '/kaggle/input/wsmte-processed-data/'

X_train = np.load(DATA_DIR + 'X_train.npy')
X_val   = np.load(DATA_DIR + 'X_val.npy')
X_test  = np.load(DATA_DIR + 'X_test.npy')

y_clf_train = np.load(DATA_DIR + 'y_clf_train.npy')
y_clf_val   = np.load(DATA_DIR + 'y_clf_val.npy')
y_clf_test  = np.load(DATA_DIR + 'y_clf_test.npy')

y_reg_train = np.load(DATA_DIR + 'y_reg_train.npy')
y_reg_val   = np.load(DATA_DIR + 'y_reg_val.npy')
y_reg_test  = np.load(DATA_DIR + 'y_reg_test.npy')

with open(DATA_DIR + 'class_weights.json') as f:
    class_weight = json.load(f)

print(f'X_train : {X_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print(f'class_weight: {class_weight}')

data = {
    'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
    'y_clf_train': y_clf_train, 'y_clf_val': y_clf_val, 'y_clf_test': y_clf_test,
    'y_reg_train': y_reg_train, 'y_reg_val': y_reg_val, 'y_reg_test': y_reg_test,
    'class_weight': class_weight,
}

## Stage 1 — Load Config G Weights
> Config G was trained in notebook 04 (concat merge, both heads, 10 seeds).
> We load the best-seed model as our Stage 1 encoder.

In [ ]:
# ── Cell 4: Load Config G model ───────────────────────────────────────────
from src.models.wsmte import build_wsmte

cfg_g = CONFIG['ablation_configs']['G']
g_model = tf.keras.models.load_model('/kaggle/input/wsmte-models/config_g_best.keras')

print('Config G model loaded.')
print(f'Input shape : {g_model.input_shape}')
print(f'Output shapes: {[o.shape for o in g_model.outputs]}')
print(f'Trainable params: {g_model.count_params():,}')

# Quick sanity check — forward pass
dummy = np.random.randn(4, CONFIG['window_size'], CONFIG['n_features']).astype('float32')
preds = g_model(dummy, training=False)
print(f'Sanity check outputs: reg={preds[0].shape}, clf={preds[1].shape}')

## Stage 2 — PSO Weight Search
> Extracts LSTM/TCN/GRU branch outputs on validation set (encoder frozen).  
> PSO searches for best [w1, w2, w3] (softmax-normalised) to maximise val accuracy.  
> n_particles=20, iterations=50, options: c1=0.5, c2=0.3, w=0.9

In [ ]:
# ── Cell 5: Run PSO weight search ─────────────────────────────────────────
from src.models.pso_weighting import run_pso_stage

print('Starting PSO Stage 2...')
print(f'  n_particles : {CONFIG["pso_n_particles"]}')
print(f'  iterations  : {CONFIG["pso_iterations"]}')

best_weights, best_cost = run_pso_stage(
    model=g_model,
    X_val=X_val,
    y_clf_val=y_clf_val,
    config=CONFIG,
)

print(f'\nPSO complete.')
print(f'Best weights  : LSTM={best_weights[0]:.4f}, TCN={best_weights[1]:.4f}, GRU={best_weights[2]:.4f}')
print(f'Sum of weights: {best_weights.sum():.6f}  (must be ~1.0)')
print(f'Best cost     : {best_cost:.6f}  (= -val_accuracy)')
print(f'Val accuracy  : {-best_cost:.4f}')

# Save PSO weights
pso_weights_dict = {
    'w_lstm': float(best_weights[0]),
    'w_tcn':  float(best_weights[1]),
    'w_gru':  float(best_weights[2]),
    'val_accuracy': float(-best_cost),
}
with open('/kaggle/working/pso_weights.json', 'w') as f:
    json.dump(pso_weights_dict, f, indent=2)
print('Saved -> /kaggle/working/pso_weights.json')

## Stage 3 — Config H Fine-tuning (30 seeds)
> Each run: builds new PSO-merge model, transfers branch weights from Config G,
> fine-tunes for up to 20 epochs (patience=5).  
> Results saved after EVERY run.

In [ ]:
# ── Cell 6: 30-seed Config H fine-tuning loop ─────────────────────────────
from src.models.pso_weighting import finetune_with_pso_weights
from src.evaluation.metrics import (
    compute_classification_metrics,
    compute_regression_metrics,
)

cfg_h = CONFIG['ablation_configs']['H']
n_runs = cfg_h['n_runs']  # 30
H_RESULTS_PATH = '/kaggle/working/ablation_results_H.csv'
PARTIAL_PATH   = '/kaggle/working/ablation_results_H_partial.csv'

all_h_rows = []
best_h_acc = -1.0
best_h_model = None

for seed in range(n_runs):
    tf.random.set_seed(seed)
    np.random.seed(seed)
    print(f'\n{"-"*50}')
    print(f'Config H | Run {seed+1}/{n_runs} | seed={seed}')

    pso_model = finetune_with_pso_weights(
        trained_model=g_model,
        pso_weights=best_weights,
        data=data,
        config=CONFIG,
    )

    # Evaluate on test set
    raw_preds = pso_model(X_test, training=False)
    reg_pred = raw_preds[0].numpy().ravel()
    clf_pred = raw_preds[1].numpy().ravel()

    clf_metrics = compute_classification_metrics(y_clf_test, clf_pred)
    reg_metrics = compute_regression_metrics(y_reg_test, reg_pred)

    row = {
        'config': 'H', 'seed': seed, 'run': seed + 1,
        **clf_metrics, **reg_metrics,
    }
    all_h_rows.append(row)

    # Save partial immediately (protect against session timeout)
    pd.DataFrame([row]).to_csv(
        PARTIAL_PATH, mode='a', index=False, header=(seed == 0)
    )
    print(f'  accuracy={clf_metrics["accuracy"]:.4f}  '
          f'auc={clf_metrics["auc"]:.4f}  '
          f'rmse={reg_metrics["rmse"]:.6f}')

    # Track best model for SHAP
    if clf_metrics['accuracy'] > best_h_acc:
        best_h_acc = clf_metrics['accuracy']
        best_h_model = pso_model
        print(f'  --> New best accuracy: {best_h_acc:.4f}')

print(f'\nAll {n_runs} Config H runs complete.')

In [ ]:
# ── Cell 7: Save Config H results + best model ────────────────────────────
results_H = pd.DataFrame(all_h_rows)
results_H.to_csv(H_RESULTS_PATH, index=False)
print(f'Saved -> {H_RESULTS_PATH}')

print(f'\nConfig H — 30-run summary:')
print(f'  Mean accuracy : {results_H["accuracy"].mean():.4f}')
print(f'  Max  accuracy : {results_H["accuracy"].max():.4f}')
print(f'  Std  accuracy : {results_H["accuracy"].std():.4f}')
print(f'  Mean AUC      : {results_H["auc"].mean():.4f}')
print(f'  Mean RMSE     : {results_H["rmse"].mean():.6f}')

# Save best Config H model
best_h_model.save('/kaggle/working/config_h_best.keras')
print(f'\nBest Config H model saved -> /kaggle/working/config_h_best.keras')
print(f'Best accuracy: {best_h_acc:.4f}')

## SHAP Analysis on Best Config H Model
> DeepSHAP / GradientExplainer on the classification head.  
> GPU-accelerated on Kaggle T4.

In [ ]:
# ── Cell 8: Run SHAP analysis ─────────────────────────────────────────────
from src.evaluation.shap_analysis import run_shap_analysis

SHAP_SAVE = '/kaggle/working/shap_summary.png'
print(f'Running SHAP GradientExplainer on {len(X_test)} test samples...')
print(f'Features: {CONFIG["feature_columns"]}')

shap_values = run_shap_analysis(
    model=best_h_model,
    X_test=X_test,
    feature_names=CONFIG['feature_columns'],
    save_path=SHAP_SAVE,
    background_size=100,
    max_display=CONFIG['n_features'],
)
print(f'SHAP done. shap_values shape: {shap_values.shape if hasattr(shap_values, "shape") else type(shap_values)}')

## Download Instructions

In [ ]:
# ── Cell 9: Summary + download list ──────────────────────────────────────
import os
downloads = [
    '/kaggle/working/ablation_results_H.csv',
    '/kaggle/working/config_h_best.keras',
    '/kaggle/working/pso_weights.json',
    '/kaggle/working/shap_summary.png',
]
print('=' * 55)
print('NOTEBOOK 05 COMPLETE')
print('=' * 55)
for p in downloads:
    size = os.path.getsize(p) // 1024 if os.path.exists(p) else 'MISSING'
    print(f'  {os.path.basename(p):35s}  {size} KB')
print()
print('Local destinations:')
print('  ablation_results_H.csv  -> ablation/')
print('  config_h_best.keras     -> results/saved_models/')
print('  shap_summary.png        -> results/figures/')
print('  pso_weights.json        -> results/')
print('Expected Kaggle runtime: 1-2 hours')